# SpaceX Launch Records Dashboard — Solved Jupyter Notebook

This notebook version solves the IBM Applied Data Science Capstone Dash lab.

It includes:

- Launch site dropdown
- Success launches pie chart callback
- Payload mass range slider
- Payload vs. success scatter chart callback
- Optional automatic insight calculations for the five dashboard questions

Run the cells from top to bottom. The final cell starts the Dash app.

## 1. Install required packages if needed

Run the next cell only if your environment does not already have these packages installed.

In [ ]:
# Uncomment and run this cell only if needed
# %pip install pandas dash plotly

## 2. Import libraries and load the SpaceX dashboard dataset

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# SpaceX dashboard dataset used in the IBM/Coursera lab
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
)
LOCAL_DATA_FILE = Path("spacex_launch_dash.csv")

# The notebook first looks for spacex_launch_dash.csv in the current folder.
# If it is not found, it tries to read the dataset from the IBM/Coursera URL.
try:
    if LOCAL_DATA_FILE.exists():
        spacex_df = pd.read_csv(LOCAL_DATA_FILE)
        print(f"Loaded local file: {LOCAL_DATA_FILE.resolve()}")
    else:
        spacex_df = pd.read_csv(DATA_URL)
        print("Loaded dataset from IBM/Coursera URL")
except Exception as error:
    raise FileNotFoundError(
        "Could not load spacex_launch_dash.csv. "
        "Download spacex_launch_dash.csv and place it in the same folder as this notebook, "
        "or make sure your internet connection is working."
    ) from error

print("Dataset shape:", spacex_df.shape)
display(spacex_df.head())

Loaded dataset from IBM/Coursera URL
Dataset shape: (56, 7)


,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [2]:
# Check available columns and launch sites
print("Columns:")
print(spacex_df.columns.tolist())

print("\nLaunch sites:")
print(sorted(spacex_df["Launch Site"].dropna().unique()))

min_payload = spacex_df["Payload Mass (kg)"].min()
max_payload = spacex_df["Payload Mass (kg)"].max()

print(f"\nPayload mass range: {min_payload} kg to {max_payload} kg")

Columns:
['Unnamed: 0', 'Flight Number', 'Launch Site', 'class', 'Payload Mass (kg)', 'Booster Version', 'Booster Version Category']

Launch sites:
['CCAFS LC-40', 'CCAFS SLC-40', 'KSC LC-39A', 'VAFB SLC-4E']

Payload mass range: 0.0 kg to 9600.0 kg


## 3. Prepare dropdown options and create the Dash application

In [3]:
# Create dropdown options dynamically from the dataset
launch_site_options = [{"label": "All Sites", "value": "ALL"}]
launch_site_options += [
    {"label": site, "value": site}
    for site in sorted(spacex_df["Launch Site"].dropna().unique())
]

# Create Dash application
app = Dash(__name__)
server = app.server

## 4. Build the dashboard layout

This solves:

- **Task 1:** Add a launch-site dropdown input component
- **Task 3:** Add a payload range slider

In [4]:
app.layout = html.Div(
    children=[
        html.H1(
            "SpaceX Launch Records Dashboard",
            style={
                "textAlign": "center",
                "color": "#503D36",
                "font-size": 40,
            },
        ),

        # TASK 1: Launch Site Dropdown
        dcc.Dropdown(
            id="site-dropdown",
            options=launch_site_options,
            value="ALL",
            placeholder="Select a Launch Site here",
            searchable=True,
        ),
        html.Br(),

        # Output for TASK 2
        html.Div(dcc.Graph(id="success-pie-chart")),
        html.Br(),

        html.P("Payload range (Kg):"),

        # TASK 3: Payload Range Slider
        dcc.RangeSlider(
            id="payload-slider",
            min=0,
            max=10000,
            step=1000,
            marks={
                0: "0",
                2500: "2500",
                5000: "5000",
                7500: "7500",
                10000: "10000",
            },
            value=[min_payload, max_payload],
        ),
        html.Br(),

        # Output for TASK 4
        html.Div(dcc.Graph(id="success-payload-scatter-chart")),
    ]
)

## 5. Add the pie-chart callback

This solves **Task 2:** render `success-pie-chart` based on the selected launch site.

In [5]:
@app.callback(
    Output(component_id="success-pie-chart", component_property="figure"),
    Input(component_id="site-dropdown", component_property="value"),
)
def get_pie_chart(entered_site):
    """Return a pie chart for all sites or for one selected launch site."""

    if entered_site == "ALL":
        # Count only successful launches for each launch site
        success_df = spacex_df[spacex_df["class"] == 1]

        fig = px.pie(
            success_df,
            names="Launch Site",
            values="class",
            title="Total Success Launches By Site",
        )
        return fig

    # For a selected site, show success vs. failure counts for that site
    filtered_df = spacex_df[spacex_df["Launch Site"] == entered_site]

    fig = px.pie(
        filtered_df,
        names="class",
        title=f"Total Success Launches for site {entered_site}",
    )
    return fig

## 6. Add the scatter-chart callback

This solves **Task 4:** render `success-payload-scatter-chart` based on selected site and payload range.

In [6]:
@app.callback(
    Output(component_id="success-payload-scatter-chart", component_property="figure"),
    [
        Input(component_id="site-dropdown", component_property="value"),
        Input(component_id="payload-slider", component_property="value"),
    ],
)
def get_payload_scatter_chart(entered_site, payload_range):
    """Return a scatter chart filtered by launch site and payload range."""

    low, high = payload_range

    payload_filtered_df = spacex_df[
        (spacex_df["Payload Mass (kg)"] >= low)
        & (spacex_df["Payload Mass (kg)"] <= high)
    ]

    if entered_site == "ALL":
        fig = px.scatter(
            payload_filtered_df,
            x="Payload Mass (kg)",
            y="class",
            color="Booster Version Category",
            title="Correlation between Payload and Success for all Sites",
        )
        return fig

    site_payload_filtered_df = payload_filtered_df[
        payload_filtered_df["Launch Site"] == entered_site
    ]

    fig = px.scatter(
        site_payload_filtered_df,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        title=f"Correlation between Payload and Success for site {entered_site}",
    )
    return fig

## 7. Optional: calculate answers for the five dashboard insight questions

The lab asks you to use the dashboard visually. This code cell also calculates the same insights from the dataset, so you can use the results in your final presentation.

In [7]:
# 1. Which site has the largest number of successful launches?
success_by_site = spacex_df[spacex_df["class"] == 1].groupby("Launch Site").size().sort_values(ascending=False)

# 2. Which site has the highest launch success rate?
success_rate_by_site = spacex_df.groupby("Launch Site")["class"].mean().sort_values(ascending=False)

# 3 and 4. Which payload ranges have the highest/lowest success rates?
# The bins below match the dashboard slider step of 1000 kg.
payload_bins = list(range(0, 11000, 1000))
spacex_df_for_bins = spacex_df.copy()
spacex_df_for_bins["Payload Range"] = pd.cut(
    spacex_df_for_bins["Payload Mass (kg)"],
    bins=payload_bins,
    include_lowest=True,
)
payload_success_rate = spacex_df_for_bins.groupby("Payload Range", observed=False)["class"].mean().dropna().sort_values(ascending=False)

# 5. Which booster version category has the highest launch success rate?
booster_success_rate = spacex_df.groupby("Booster Version Category")["class"].mean().sort_values(ascending=False)

print("1. Site with largest successful launches:")
print(success_by_site.head(1))

print("\n2. Site with highest launch success rate:")
print(success_rate_by_site.head(1))

print("\n3. Payload range(s) with highest success rate:")
print(payload_success_rate.head(3))

print("\n4. Payload range(s) with lowest success rate:")
print(payload_success_rate.tail(3))

print("\n5. Booster version category with highest launch success rate:")
print(booster_success_rate.head(5))

1. Site with largest successful launches:
Launch Site
KSC LC-39A    10
dtype: int64

2. Site with highest launch success rate:
Launch Site
KSC LC-39A    0.769231
Name: class, dtype: float64

3. Payload range(s) with highest success rate:
Payload Range
(3000.0, 4000.0]     0.727273
(9000.0, 10000.0]    0.600000
(2000.0, 3000.0]     0.500000
Name: class, dtype: float64

4. Payload range(s) with lowest success rate:
Payload Range
(1000.0, 2000.0]    0.333333
(-0.001, 1000.0]    0.200000
(6000.0, 7000.0]    0.000000
Name: class, dtype: float64

5. Booster version category with highest launch success rate:
Booster Version Category
B5      1.000000
FT      0.666667
B4      0.545455
v1.1    0.066667
v1.0    0.000000
Name: class, dtype: float64


## 8. Run the Dash app

Run the next cell and open the URL shown in the output.

For local Jupyter on your computer, use:

```text
http://127.0.0.1:8050/
```

To stop the app, interrupt the kernel or press the stop button in Jupyter.

In [8]:
# For local Jupyter Notebook/JupyterLab
HOST = "127.0.0.1"
PORT = 8050

# If you are running in the Coursera/Skills Network Cloud IDE instead,
# you can change HOST to "0.0.0.0".

try:
    app.run(jupyter_mode="external", host=HOST, port=PORT, debug=False)
except TypeError:
    # Fallback for older Dash versions that do not support jupyter_mode
    app.run_server(host=HOST, port=PORT, debug=False)

Dash app running on http://127.0.0.1:8050/
